# BAB 3 — Data Preprocessing
## Dari polygon mentah menjadi dataset ML

## 3.1 Validasi dan pemilihan polygon

**Pertanyaan:** dari 2823 polygon mentah, mana yang valid, berapa yang terbuang per kode, dan 50 per kelas mana yang terpilih?

In [1]:
# DATA AKTUAL PROJECT — 3.1a V1-V4 + V6 dari file mentah, V5 dari artefak Bab 2 (aturan sama).
import os

import geopandas as gpd
import pandas as pd
from shapely.validation import explain_validity

RANDOM_STATE, N_PER_CLASS, MIN_VALID_PIXELS = 42, 50, 1
EXPECTED = {1: "Sawah", 2: "Bangunan / Permukiman", 3: "Mangrove",
            4: "Lahan hijau", 5: "Laut", 6: "Danau"}
os.makedirs("../outputs/tables", exist_ok=True)
g = gpd.read_file("../data/training_samples_jawa_timur.geojson")
print(f"DATA AKTUAL PROJECT: n_mentah = {len(g)}")
g["kode"] = "VALID"
g["alasan"] = ""
m = g.geometry.isna() | g.geometry.is_empty
g.loc[m, "kode"] = "V1"; g.loc[m, "alasan"] = "geometri null/kosong"
dup = g.duplicated(subset=["fid"], keep=False) & (g["kode"] == "VALID")
g.loc[dup, "kode"] = "V1"; g.loc[dup, "alasan"] = "fid duplikat"
ok = g["kode"] == "VALID"
t = g.loc[ok, "geometry"].geom_type
g.loc[ok & ~t.isin(["Polygon", "MultiPolygon"]), "kode"] = "V2"
g.loc[g["kode"] == "V2", "alasan"] = "tipe bukan Polygon/MultiPolygon"
ok = g["kode"] == "VALID"
vg = g.loc[ok, "geometry"].is_valid
m3 = ok & ~vg.reindex(g.index, fill_value=True).fillna(False).astype(bool)
g.loc[m3, "kode"] = "V3"
for i in g.index[m3]:
    try:
        g.at[i, "alasan"] = str(explain_validity(g.at[i, "geometry"]))[:120]
    except Exception:
        g.at[i, "alasan"] = "invalid (alasan tidak terbaca)"
ok = g["kode"] == "VALID"
kons = g["class_id"].isin([1, 2, 3, 4, 5, 6]) & (g["class_name"] == g["class_id"].map(EXPECTED))
g.loc[ok & ~kons, "kode"] = "V4"
g.loc[g["kode"] == "V4", "alasan"] = "label tidak lengkap/konsisten"
gm = g.loc[g["kode"] == "VALID", ["fid", "class_id", "geometry"]].copy()
gm = gm[gm.geometry.notna() & ~gm.geometry.is_empty].to_crs("EPSG:32749")
gm["area"] = gm.geometry.area
geom_of = dict(zip(gm["fid"], gm["geometry"]))
area_of = dict(zip(gm["fid"], gm["area"]))
v6 = set()
if len(gm):
    cand = gpd.sjoin(gm, gm, predicate="intersects", how="inner", lsuffix="l", rsuffix="r")
    cand = cand[(cand["fid_l"] != cand["fid_r"]) & (cand["class_id_l"] != cand["class_id_r"])]
    for fl, fr in zip(cand["fid_l"].tolist(), cand["fid_r"].tolist()):
        if geom_of[fl].intersection(geom_of[fr]).area > 0.05 * float(area_of[fl]):
            v6.add(fl)
print(f"DATA AKTUAL PROJECT: V6 = {len(v6)} (logika sama, pencarian via dict agar cepat)")
g.loc[g["fid"].isin(v6) & (g["kode"] == "VALID"), "kode"] = "V6"
g.loc[g["kode"] == "V6", "alasan"] = "overlap lintas kelas > 0.05"
stat = pd.read_csv("../outputs/tables/polygon_pixel_stats.csv")
g = g.merge(stat, on="fid", how="left", validate="one_to_one")
print(f"DATA AKTUAL PROJECT: V1={(g['kode'] == 'V1').sum()} V2={(g['kode'] == 'V2').sum()} " +
      f"V3={(g['kode'] == 'V3').sum()} V4={(g['kode'] == 'V4').sum()} V6={(g['kode'] == 'V6').sum()}")
# V5 final: kode VALID tetapi < MIN_VALID_PIXELS pixel (termasuk 107 gagal-baca di luar extent).
mask_v5 = (g["kode"] == "VALID") & (g["n_valid"] < MIN_VALID_PIXELS)
g.loc[mask_v5, "kode"] = "V5"
g.loc[mask_v5, "alasan"] = "<1 pixel valid (" + g.loc[mask_v5, "status_v5"].astype(str) + ")"
print(f"DATA AKTUAL PROJECT: V5 = {int((g['kode'] == 'V5').sum())} (arti: n_valid<1 atau gagal baca)")
g.to_pickle("../outputs/tables/validity_final.pkl")

DATA AKTUAL PROJECT: n_mentah = 2823


DATA AKTUAL PROJECT: V6 = 47 (logika sama, pencarian via dict agar cepat)


DATA AKTUAL PROJECT: V1=0 V2=0 V3=0 V4=0 V6=47
DATA AKTUAL PROJECT: V5 = 2008 (arti: n_valid<1 atau gagal baca)


In [2]:
# DATA AKTUAL PROJECT — 3.1b tabel sebelum|dibuang|pool|terpilih|kuota + seleksi acak 50/kelas.
import pandas as pd

g = pd.read_pickle("../outputs/tables/validity_final.pkl")
pool = g[(g["kode"] == "VALID") & (g["n_valid"] >= 1)].copy()
rows = []
sel_parts = []
for cid in [1, 2, 3, 4, 5, 6]:
    sub = g[g["class_id"] == cid]
    d = {k: int(((sub["kode"] == k)).sum()) for k in ["V1", "V2", "V3", "V4", "V5", "V6"]}
    pv = pool[pool["class_id"] == cid]
    n_ambil = min(N_PER_CLASS, len(pv))
    s = pv.sample(n=n_ambil, random_state=RANDOM_STATE) if n_ambil else pv
    sel_parts.append(s)
    rows.append({"class_id": cid, "class_name": sub["class_name"].iloc[0], "sebelum": len(sub),
                 **{f"buang_{k}": d[k] for k in d}, "pool_valid": len(pv),
                 "terpilih": len(s), "valid_tidak_terpilih_kuota": len(pv) - len(s)})
tab = pd.DataFrame(rows)
print("DATA AKTUAL PROJECT: sebelum | dibuang per kode | pool | terpilih | kuota =")
print(tab.to_string(index=False))
sel = pd.concat(sel_parts).sort_values("fid").reset_index(drop=True)
print(f"DATA AKTUAL PROJECT: total_terpilih = {len(sel)} (target 300 bila semua kelas >= 50)")
sel_geo = sel.merge(g[["fid"]], on="fid")
gusel = g[g["fid"].isin(sel["fid"])].copy()
gusel.to_file("../outputs/tables/polygon_selected.geojson", driver="GeoJSON")
sel[["fid", "class_id", "class_name", "n_valid"]].to_csv("../outputs/tables/polygon_selected.csv", index=False)
exc_inv = g[~g["fid"].isin(sel["fid"]) & (g["kode"] != "VALID")][["fid", "class_id", "class_name", "kode", "alasan"]].copy()
exc_inv["kategori"] = "tidak valid"
exc_kuota = pool[~pool["fid"].isin(sel["fid"])][["fid", "class_id", "class_name"]].copy()
exc_kuota["kode"] = "KUOTA"
exc_kuota["alasan"] = "valid tetapi tidak terambil acak 50/kelas"
exc_kuota["kategori"] = "valid tidak terpilih (kuota)"
exc = pd.concat([exc_inv, exc_kuota]).sort_values("fid")
exc[["fid", "class_id", "class_name", "kategori", "kode", "alasan"]].to_csv("../outputs/tables/polygon_excluded.csv", index=False)
print(f"DATA AKTUAL PROJECT: excluded tidak_valid={len(exc_inv)}, kuota={len(exc_kuota)}")
r367 = g[g["fid"] == 367]
if len(r367):
    r = r367.iloc[0]
    st = "TERPILIH" if 367 in set(sel["fid"]) else ("tidak valid: " + r["kode"] if r["kode"] != "VALID" else "valid tetapi tidak terambil kuota")
    print(f"DATA AKTUAL PROJECT: FID 367 class {int(r['class_id'])} {r['class_name']} kode={r['kode']} n_valid={int(r['n_valid'])} -> {st}")
else:
    print("DATA AKTUAL PROJECT: FID 367 tidak tersedia di file")

DATA AKTUAL PROJECT: sebelum | dibuang per kode | pool | terpilih | kuota =
 class_id            class_name  sebelum  buang_V1  buang_V2  buang_V3  buang_V4  buang_V5  buang_V6  pool_valid  terpilih  valid_tidak_terpilih_kuota
        1                 Sawah      500         0         0         0         0       407         5          88        50                          38
        2 Bangunan / Permukiman      500         0         0         0         0       246         2         252        50                         202
        3              Mangrove      500         0         0         0         0       357        35         108        50                          58
        4           Lahan hijau      500         0         0         0         0       394         5         101        50                          51
        5                  Laut      500         0         0         0         0       494         0           6         6                           0
        6         

DATA AKTUAL PROJECT: excluded tidak_valid=2055, kuota=512
DATA AKTUAL PROJECT: FID 367 class 1 Sawah kode=VALID n_valid=40 -> TERPILIH


**Interpretasi 3.1:** pembuang utama adalah V5 = 2008 polygon (71%): 1901 tanpa pixel valid + 107 gagal baca di luar extent (rekonsiliasi dengan Bab 2: 1901 + 107 = 2008). V1–V4 nihil, V6 = 47. Terpilih 256 polygon (50 × 5 kelas + 6 Laut); 512 polygon valid terbuang kuota; FID 367 TERPILIH (Sawah, 40 pixel). Target 300 tidak tercapai karena Laut — dilaporkan, bukan diisi data baru.

## 3.2 Ekstraksi pixel + capping 500

**Pertanyaan:** berapa pixel per polygon sebelum/sesudah cap, berapa polygon terkena cap, dan bagaimana indeks dihitung?

In [3]:
# DATA AKTUAL PROJECT — 3.2 ekstraksi centroid + cap 500 + indeks (nodata & 0-denom ditangani).
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds

MAXP = 500
BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
g = gpd.read_file("../outputs/tables/polygon_selected.geojson")
stat = pd.read_csv("../outputs/tables/polygon_pixel_stats.csv")
nmap = dict(zip(stat["fid"], stat["n_valid"]))
rng = np.random.RandomState(RANDOM_STATE)
recs, pix = [], {}
n_capped = 0
with rasterio.open("../data/s2_jatim.tif") as src:
    nd = src.nodata
    gu = g.to_crs(src.crs).sort_values("fid").reset_index(drop=True)
    for _, row in gu.iterrows():
        fid, cid = int(row["fid"]), int(row["class_id"])
        geom = row["geometry"]
        w = from_bounds(*geom.bounds, src.transform).crop(src.height, src.width)
        arr = src.read(window=w, boundless=True, fill_value=nd)
        m = geometry_mask([geom], out_shape=(arr.shape[1], arr.shape[2]),
                            transform=src.window_transform(w), invert=True, all_touched=False)
        vals = arr[:, m & (arr != nd).all(axis=0)].T
        n_before = len(vals)
        capped = n_before > MAXP
        if capped:
            vals = vals[rng.choice(n_before, MAXP, replace=False)]
            n_capped += 1
        pix[fid] = vals
        recs.append((fid, cid, row["class_name"], n_before, len(vals), bool(capped)))
cap = pd.DataFrame(recs, columns=["fid", "class_id", "class_name", "n_sebelum", "n_sesudah", "terkena_cap"])
cap.to_csv("../outputs/tables/pixel_cap_stats.csv", index=False)
print(f"DATA AKTUAL PROJECT: polygon_terkena_cap = {n_capped} dari {len(cap)}")
print(f"DATA AKTUAL PROJECT: pixel sebelum = {int(cap['n_sebelum'].sum())}, sesudah = {int(cap['n_sesudah'].sum())}")
# Indeks pada sampel sesudah-cap; penyebut nol -> NaN (dihitung).
frames = []
for fid, vals in pix.items():
    df = pd.DataFrame(vals, columns=BANDS).astype(float)
    df.insert(0, "fid", fid)
    frames.append(df)
px = pd.concat(frames, ignore_index=True)
den_v = px["B08"] + px["B04"]
den_w = px["B03"] + px["B08"]
den_b = px["B11"] + px["B08"]
px["NDVI"] = np.where(den_v != 0, (px["B08"] - px["B04"]) / np.where(den_v == 0, 1, den_v), np.nan)
px["NDWI"] = np.where(den_w != 0, (px["B03"] - px["B08"]) / np.where(den_w == 0, 1, den_w), np.nan)
px["NDBI"] = np.where(den_b != 0, (px["B11"] - px["B08"]) / np.where(den_b == 0, 1, den_b), np.nan)
print(f"DATA AKTUAL PROJECT: pixel_total = {len(px)}, NDVI_NaN = {int(px['NDVI'].isna().sum())}, " +
      f"NDWI_NaN = {int(px['NDWI'].isna().sum())}, NDBI_NaN = {int(px['NDBI'].isna().sum())}")
px.to_pickle("../outputs/tables/pixels_selected.pkl")

DATA AKTUAL PROJECT: polygon_terkena_cap = 62 dari 256
DATA AKTUAL PROJECT: pixel sebelum = 345893, sesudah = 51508


DATA AKTUAL PROJECT: pixel_total = 51508, NDVI_NaN = 0, NDWI_NaN = 0, NDBI_NaN = 0


**Interpretasi 3.2:** 62 dari 256 polygon (>500 pixel) terkena cap; total 345.893 → 51.508 pixel (±15% dipertahankan). Penyebut nol pada indeks: 0 pixel terdampak — semua NDVI/NDWI/NDBI terdefinisi.

## 3.3 Train/test split per polygon

**Mengapa per polygon, bukan per pixel (CONTOH ILUSTRASI — bukan data project):**

```text
BENAR:   Polygon A (50 pixel) -> SEMUA ke TRAIN;  Polygon B (40 pixel) -> SEMUA ke TEST
DILARANG: Polygon A -> 40 pixel train + 10 pixel test (pixel bertetangga sangat mirip
         -> spatial/data leakage -> metrik terlalu optimistis)
```

Split 80/20 polygon, `random_state=42`, stratified per kelas. Jujur: split per polygon belum menjamin bebas autokorelasi spasial antar polygon bertetangga — jarak train–test dilaporkan sebagai informasi, bukan untuk mengubah split.

In [4]:
# DATA AKTUAL PROJECT — 3.3 split stratified + bukti irisan kosong + jarak sebagai informasi.
import pandas as pd
from sklearn.model_selection import train_test_split

sel = pd.read_csv("../outputs/tables/polygon_selected.csv")
tr, te = train_test_split(sel, test_size=0.20, random_state=RANDOM_STATE, stratify=sel["class_id"])
tr["split"], te["split"] = "train", "test"
sp = pd.concat([tr, te])
sp[["fid", "class_id", "class_name", "split"]].to_csv("../outputs/tables/polygon_split.csv", index=False)
print(f"DATA AKTUAL PROJECT: polygon train = {len(tr)}, test = {len(te)} (total {len(sel)})")
tab = sp.groupby(["class_id", "class_name", "split"]).size().unstack(fill_value=0)
print("DATA AKTUAL PROJECT: class_id | class_name | training_polygon | testing_polygon =")
print(tab.to_string())
print(f"DATA AKTUAL PROJECT: irisan FID train-test = {sorted(set(tr['fid']) & set(te['fid']))} (kosong={len(set(tr['fid']) & set(te['fid'])) == 0})")
print("DATA AKTUAL PROJECT: contoh FID|class|split (367 bila terpilih, +1 per kelas) =")
shown = set()
if 367 in set(sel["fid"]):
    r = sp[sp["fid"] == 367].iloc[0]
    print(f"  FID 367 | {int(r['class_id'])} {r['class_name']} | {r['split']}")
    shown.add(int(r["class_id"]))
for cid in [1, 2, 3, 4, 5, 6]:
    if cid in shown:
        continue
    r = sp[sp["class_id"] == cid].iloc[0]
    print(f"  FID {int(r['fid'])} | {cid} {r['class_name']} | {r['split']}")
# Jarak train-test sebagai INFORMASI (metric CRS, centroid).
import geopandas as gpd
gsel = gpd.read_file("../outputs/tables/polygon_selected.geojson").to_crs("EPSG:32749")
gsel = gsel.merge(sp[["fid", "split"]], on="fid")
ctr_tr = gsel[gsel["split"] == "train"].geometry.centroid
for _, r in gsel[gsel["split"] == "test"].iterrows():
    dmin = float(ctr_tr.distance(r["geometry"].centroid).min())
    print(f"DATA AKTUAL PROJECT: test FID {int(r['fid'])} class {int(r['class_id'])} jarak_ke_train_terdekat_m = {dmin:.0f}")

DATA AKTUAL PROJECT: polygon train = 204, test = 52 (total 256)
DATA AKTUAL PROJECT: class_id | class_name | training_polygon | testing_polygon =
split                           test  train
class_id class_name                        
1        Sawah                    10     40
2        Bangunan / Permukiman    10     40
3        Mangrove                 10     40
4        Lahan hijau              10     40
5        Laut                      2      4
6        Danau                    10     40
DATA AKTUAL PROJECT: irisan FID train-test = [] (kosong=True)
DATA AKTUAL PROJECT: contoh FID|class|split (367 bila terpilih, +1 per kelas) =
  FID 367 | 1 Sawah | train
  FID 698 | 2 Bangunan / Permukiman | train
  FID 1440 | 3 Mangrove | train
  FID 1887 | 4 Lahan hijau | train
  FID 2044 | 5 Laut | train
  FID 2616 | 6 Danau | train


DATA AKTUAL PROJECT: test FID 72 class 1 jarak_ke_train_terdekat_m = 4387
DATA AKTUAL PROJECT: test FID 99 class 1 jarak_ke_train_terdekat_m = 3841
DATA AKTUAL PROJECT: test FID 154 class 1 jarak_ke_train_terdekat_m = 11603
DATA AKTUAL PROJECT: test FID 176 class 1 jarak_ke_train_terdekat_m = 4202
DATA AKTUAL PROJECT: test FID 183 class 1 jarak_ke_train_terdekat_m = 11306
DATA AKTUAL PROJECT: test FID 189 class 1 jarak_ke_train_terdekat_m = 14659
DATA AKTUAL PROJECT: test FID 310 class 1 jarak_ke_train_terdekat_m = 6268
DATA AKTUAL PROJECT: test FID 345 class 1 jarak_ke_train_terdekat_m = 1774
DATA AKTUAL PROJECT: test FID 348 class 1 jarak_ke_train_terdekat_m = 288
DATA AKTUAL PROJECT: test FID 393 class 1 jarak_ke_train_terdekat_m = 5000
DATA AKTUAL PROJECT: test FID 530 class 2 jarak_ke_train_terdekat_m = 4605
DATA AKTUAL PROJECT: test FID 537 class 2 jarak_ke_train_terdekat_m = 766
DATA AKTUAL PROJECT: test FID 544 class 2 jarak_ke_train_terdekat_m = 5972
DATA AKTUAL PROJECT: test 

**Interpretasi 3.3:** train 204 / test 52 polygon; irisan FID terbukti kosong (`[]`). Stratifikasi terjaga (10 per kelas; Laut 2 test / 4 train). FID 367 di train. Jarak test–train terdekat ada yang <500 m (mis. Danau 173 m, Sawah 288 m): split per polygon tidak menghilangkan autokorelasi spasial antar tetangga — dicatat sebagai informasi; ketidakpastian dilaporkan di Bab 5.

## 3.4 Dataset pixel-based

**Pertanyaan:** berapa baris train/test per kelas, dan mengapa berbeda walau polygon-nya sama-sama 50 (atau kurang)?

In [5]:
# DATA AKTUAL PROJECT — 3.4 training_pixel / testing_pixel (FID sebagai group).
import pandas as pd

px = pd.read_pickle("../outputs/tables/pixels_selected.pkl")
sp = pd.read_csv("../outputs/tables/polygon_split.csv")
px = px.merge(sp[["fid", "class_id", "class_name", "split"]], on="fid", validate="many_to_one")
FEATS = ["B02", "B03", "B04", "B08", "B11", "B12", "NDVI", "NDWI", "NDBI"]
for s_ in ["train", "test"]:
    d = px[px["split"] == s_][["fid", "class_id", "class_name"] + FEATS]
    d.to_csv(f"../outputs/tables/{'training' if s_ == 'train' else 'testing'}_pixel.csv", index=False)
    t = d.groupby(["class_id", "class_name"]).agg(polygon_count=("fid", "nunique"), pixel_count=("fid", "size"))
    t["mean_pixel_per_polygon"] = (t["pixel_count"] / t["polygon_count"]).round(1)
    print(f"DATA AKTUAL PROJECT: {'training' if s_ == 'train' else 'testing'}_pixel =")
    print(t.to_string())
print(f"DATA AKTUAL PROJECT: irisan FID pixel train-test kosong = " +
      f"{len(set(px[px['split'] == 'train']['fid']) & set(px[px['split'] == 'test']['fid'])) == 0}")
print("DATA AKTUAL PROJECT: class_weight TIDAK dibenamkan di data; diputuskan setara di Bab 4")

DATA AKTUAL PROJECT: training_pixel =
                                polygon_count  pixel_count  mean_pixel_per_polygon
class_id class_name                                                               
1        Sawah                             40        12309                   307.7
2        Bangunan / Permukiman             40         7742                   193.6
3        Mangrove                          40         7179                   179.5
4        Lahan hijau                       40         9588                   239.7
5        Laut                               4         1196                   299.0
6        Danau                             40         2062                    51.6


DATA AKTUAL PROJECT: testing_pixel =
                                polygon_count  pixel_count  mean_pixel_per_polygon
class_id class_name                                                               
1        Sawah                             10         3172                   317.2
2        Bangunan / Permukiman             10         2279                   227.9
3        Mangrove                          10         1532                   153.2
4        Lahan hijau                       10         2770                   277.0
5        Laut                               2          879                   439.5
6        Danau                             10          800                    80.0
DATA AKTUAL PROJECT: irisan FID pixel train-test kosong = True
DATA AKTUAL PROJECT: class_weight TIDAK dibenamkan di data; diputuskan setara di Bab 4


**Interpretasi 3.4:** train 40.076 vs test 11.432 baris. Walau lima kelas sama-sama 50 polygon, pixel berbeda jauh (train Sawah 12.309 vs Danau 2.062) karena ukuran polygon; capping meredam tetapi tidak menyamakan. Imbalance ini TIDAK dikoreksi di data — penanganan (mis. class_weight) diputuskan setara untuk kedua metode di Bab 4.

## 3.5 Dataset polygon-mean

**Pertanyaan:** berapa baris train/test, dan bagaimana ukurannya dibanding pixel-based (tanpa memilih pemenang)?

In [6]:
# DATA AKTUAL PROJECT — 3.5 mean dari SEMUA pixel valid (tanpa cap) + indeks dari mean band.
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds

BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
g = gpd.read_file("../outputs/tables/polygon_selected.geojson")
sp = pd.read_csv("../outputs/tables/polygon_split.csv")
rows = []
with rasterio.open("../data/s2_jatim.tif") as src:
    nd = src.nodata
    for _, row in g.to_crs(src.crs).sort_values("fid").iterrows():
        fid = int(row["fid"])
        w = from_bounds(*row["geometry"].bounds, src.transform).crop(src.height, src.width)
        arr = src.read(window=w, boundless=True, fill_value=nd)
        m = geometry_mask([row["geometry"]], out_shape=(arr.shape[1], arr.shape[2]),
                            transform=src.window_transform(w), invert=True, all_touched=False)
        vals = arr[:, m & (arr != nd).all(axis=0)].T.astype(float)
        mu = vals.mean(axis=0)
        d = dict(zip(BANDS, mu))
        rows.append((fid, int(row["class_id"]), row["class_name"], len(vals), *mu,
                       (d["B08"] - d["B04"]) / (d["B08"] + d["B04"]) if (d["B08"] + d["B04"]) != 0 else np.nan,
                       (d["B03"] - d["B08"]) / (d["B03"] + d["B08"]) if (d["B03"] + d["B08"]) != 0 else np.nan,
                       (d["B11"] - d["B08"]) / (d["B11"] + d["B08"]) if (d["B11"] + d["B08"]) != 0 else np.nan))
cols = ["fid", "class_id", "class_name", "n_pixels"] + [f"mean_{b}" for b in BANDS] + ["NDVI", "NDWI", "NDBI"]
mn = pd.DataFrame(rows, columns=cols).merge(sp[["fid", "split"]], on="fid")
for s_ in ["train", "test"]:
    mn[mn["split"] == s_].drop(columns=["split"]).to_csv(f"../outputs/tables/{'training' if s_ == 'train' else 'testing'}_mean.csv", index=False)
t = mn.groupby(["class_id", "class_name", "split"]).size().unstack(fill_value=0)
print("DATA AKTUAL PROJECT: class_id | class_name | training_polygon | testing_polygon =")
print(t.to_string())
print("DATA AKTUAL PROJECT: contoh aktual FID|class|mean =")
e = mn[mn["fid"] == 367].iloc[0] if 367 in set(mn["fid"]) else mn.iloc[0]
print(f"  FID {int(e['fid'])} | {int(e['class_id'])} {e['class_name']} | " + " ".join(f"{c}={e[c]:.1f}" for c in [f"mean_{b}" for b in BANDS]))
px_tr = pd.read_csv("../outputs/tables/training_pixel.csv")
print(f"DATA AKTUAL PROJECT: ukuran train pixel-based = {len(px_tr)} baris vs polygon-mean = {int((mn['split'] == 'train').sum())} baris")

DATA AKTUAL PROJECT: class_id | class_name | training_polygon | testing_polygon =
split                           test  train
class_id class_name                        
1        Sawah                    10     40
2        Bangunan / Permukiman    10     40
3        Mangrove                 10     40
4        Lahan hijau              10     40
5        Laut                      2      4
6        Danau                    10     40
DATA AKTUAL PROJECT: contoh aktual FID|class|mean =
  FID 367 | 1 Sawah | mean_B02=359.0 mean_B03=579.9 mean_B04=502.2 mean_B08=2560.1 mean_B11=1846.4 mean_B12=1128.2
DATA AKTUAL PROJECT: ukuran train pixel-based = 40076 baris vs polygon-mean = 204 baris


**Interpretasi 3.5:** train pixel-based 40.076 baris vs polygon-mean 204 baris (±196:1). Mean FID 367 identik dengan Bab 2 (40 pixel, tanpa cap) — konsisten. Tanpa memilih pemenang: perbandingan dijawab Bab 4–5.

## 3.6 Feature engineering

Sembilan feature dipakai SAMA oleh semua model (Bab 4): 6 band + 3 indeks. Band = nilai reflektansi skala 0–10000 (L2A, dari data 2.3); indeks tak berdimensi (-1…1). Yang tidak dipakai tidak ditulis di sini.

In [7]:
# DATA AKTUAL PROJECT — 3.6 contoh hitung langkah-demi-langkah dari data nyata.
import pandas as pd

tr = pd.read_csv("../outputs/tables/training_pixel.csv")
r = tr.iloc[0]
print(f"DATA AKTUAL PROJECT: contoh pixel nyata FID {int(r['fid'])} {r['class_name']}: " +
      " ".join(f"{b}={int(r[b])}" for b in ["B02", "B03", "B04", "B08", "B11", "B12"]))
b02, b03, b04, b08, b11 = (float(r[b]) for b in ["B02", "B03", "B04", "B08", "B11"])
print(f"  NDVI = ({b08:.0f}-{b04:.0f})/({b08:.0f}+{b04:.0f}) = {(b08 - b04) / (b08 + b04):.3f} (tersimpan {r['NDVI']:.3f}); positif->vegetasi")
print(f"  NDWI = ({b03:.0f}-{b08:.0f})/({b03:.0f}+{b08:.0f}) = {(b03 - b08) / (b03 + b08):.3f} (tersimpan {r['NDWI']:.3f}); positif->air")
print(f"  NDBI = ({b11:.0f}-{b08:.0f})/({b11:.0f}+{b08:.0f}) = {(b11 - b08) / (b11 + b08):.3f} (tersimpan {r['NDBI']:.3f}); positif->terbangun")

DATA AKTUAL PROJECT: contoh pixel nyata FID 0 Sawah: B02=276 B03=501 B04=366 B08=2341 B11=1824 B12=1117
  NDVI = (2341-366)/(2341+366) = 0.730 (tersimpan 0.730); positif->vegetasi
  NDWI = (501-2341)/(501+2341) = -0.647 (tersimpan -0.647); positif->air
  NDBI = (1824-2341)/(1824+2341) = -0.124 (tersimpan -0.124); positif->terbangun


## 3.7 Hasil perantara tersimpan

**Pertanyaan:** file apa saja yang dihasilkan tahap ini?

In [8]:
# DATA AKTUAL PROJECT — 3.7 verifikasi artefak + konfigurasi tercatat.
import json
import os

import pandas as pd

cfg = {"RANDOM_STATE": 42, "TEST_SIZE": 0.20, "N_PER_CLASS": 50, "MAX_PIXELS_PER_POLYGON": 500,
       "MIN_VALID_PIXELS": 1, "pixel_rule": "centroid (all_touched=False)",
       "cap_rng": "RandomState(42) urut fid", "mean_index_def": "indeks dari mean band", "split": "per polygon stratified"}
with open("../outputs/tables/preprocessing_config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)
for fn in ["polygon_selected.geojson", "polygon_selected.csv", "polygon_excluded.csv", "polygon_split.csv",
             "pixel_cap_stats.csv", "training_pixel.csv", "testing_pixel.csv",
             "training_mean.csv", "testing_mean.csv", "preprocessing_config.json"]:
    p = os.path.join("../outputs/tables", fn)
    info = f"baris={len(pd.read_csv(p))}" if p.endswith(".csv") else "ada"
    print(f"DATA AKTUAL PROJECT: {fn} ada={os.path.exists(p)} {info if os.path.exists(p) else ''}")

DATA AKTUAL PROJECT: polygon_selected.geojson ada=True ada
DATA AKTUAL PROJECT: polygon_selected.csv ada=True baris=256
DATA AKTUAL PROJECT: polygon_excluded.csv ada=True baris=2567
DATA AKTUAL PROJECT: polygon_split.csv ada=True baris=256
DATA AKTUAL PROJECT: pixel_cap_stats.csv ada=True baris=256


DATA AKTUAL PROJECT: training_pixel.csv ada=True baris=40076
DATA AKTUAL PROJECT: testing_pixel.csv ada=True baris=11432
DATA AKTUAL PROJECT: training_mean.csv ada=True baris=204
DATA AKTUAL PROJECT: testing_mean.csv ada=True baris=52
DATA AKTUAL PROJECT: preprocessing_config.json ada=True ada


## Checkpoint Tahap 3

- Tabel pengecualian lengkap: `polygon_excluded.csv` 2.567 baris (2.055 tidak valid + 512 kuota).
- Maksimum 50 terpilih per kelas (Laut: 6 yang valid-berpixel, dilaporkan).
- Irisan FID train/test terbukti kosong (`[]`, kedua metode memakai FID yang sama).
- 10 artefak tersimpan dan terverifikasi di `outputs/tables/`.